In [1]:
import pandas as pd

runs = pd.read_csv("../data/raw/production_runs.csv")
sensors = pd.read_csv("../data/raw/sensor_readings.csv")
defects = pd.read_csv("../data/raw/defect_events.csv")
machines = pd.read_csv("../data/raw/machines.csv")

print("Production runs:", runs.shape)
print("Sensor readings:", sensors.shape)
print("Defect events:", defects.shape)
print("Machines:", machines.shape)

Production runs: (60000, 13)
Sensor readings: (60000, 11)
Defect events: (59959, 5)
Machines: (20, 5)


In [2]:
runs.head()

,run_id,timestamp,line_id,machine_id,shift_id,material_id,operator_id,planned_units,produced_units,good_units,defective_units,yield_rate,defect_rate
0,R000001,2025-12-29,L01,M01,Morning,MAT07,OP018,432,432,394,38,0.912037,0.087963
1,R000002,2026-07-23,L02,M20,Evening,MAT03,OP024,381,381,368,13,0.965879,0.034121
2,R000003,2025-12-17,L03,M11,Night,MAT05,OP005,287,287,266,21,0.926829,0.073171
3,R000004,2026-06-01,L01,M01,Night,MAT04,OP011,436,436,392,44,0.899083,0.100917
4,R000005,2026-03-14,L02,M06,Evening,MAT09,OP015,366,366,351,15,0.959016,0.040984


In [3]:
runs.info()


<class 'pandas.DataFrame'>
RangeIndex: 60000 entries, 0 to 59999
Data columns (total 13 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   run_id           60000 non-null  str    
 1   timestamp        60000 non-null  str    
 2   line_id          60000 non-null  str    
 3   machine_id       60000 non-null  str    
 4   shift_id         60000 non-null  str    
 5   material_id      60000 non-null  str    
 6   operator_id      60000 non-null  str    
 7   planned_units    60000 non-null  int64  
 8   produced_units   60000 non-null  int64  
 9   good_units       60000 non-null  int64  
 10  defective_units  60000 non-null  int64  
 11  yield_rate       60000 non-null  float64
 12  defect_rate      60000 non-null  float64
dtypes: float64(2), int64(4), str(7)
memory usage: 6.0 MB


In [4]:
import numpy as np

np.random.seed(99)  # different seed so this messiness is separate from data generation

# Always work on a COPY — never modify the original loaded data directly
runs_dirty = runs.copy()
sensors_dirty = sensors.copy()

# ---- Issue 1: Missing values in sensor readings ----
# Simulate sensor malfunctions: randomly blank out ~2% of temperature and vibration readings
missing_idx_temp = sensors_dirty.sample(frac=0.02, random_state=1).index
sensors_dirty.loc[missing_idx_temp, "temperature"] = np.nan

missing_idx_vib = sensors_dirty.sample(frac=0.015, random_state=2).index
sensors_dirty.loc[missing_idx_vib, "vibration"] = np.nan

print("Missing temperature values:", sensors_dirty["temperature"].isna().sum())
print("Missing vibration values:", sensors_dirty["vibration"].isna().sum())

Missing temperature values: 1200
Missing vibration values: 900


In [6]:
# ---- Issue 2: Duplicate rows ----
# Simulate a data pipeline glitch that accidentally re-logged some runs
duplicate_rows = runs_dirty.sample(n=50, random_state=3)
runs_dirty = pd.concat([runs_dirty, duplicate_rows], ignore_index=True)

print("Total rows after duplication:", len(runs_dirty))
print("Duplicate run_ids:", runs_dirty["run_id"].duplicated().sum())

# ---- Issue 3: Inconsistent categorical values ----
# Simulate inconsistent data entry: shift names typed differently across systems
inconsistent_idx = runs_dirty.sample(frac=0.01, random_state=4).index
shift_variants = {"Morning": "morning ", "Evening": "EVENING", "Night": "Nite"}
runs_dirty.loc[inconsistent_idx, "shift_id"] = runs_dirty.loc[inconsistent_idx, "shift_id"].map(
    lambda s: shift_variants.get(s, s)
)

print("\nUnique shift_id values now:", runs_dirty["shift_id"].unique())

# ---- Issue 4: Outliers / impossible sensor values ----
sensors_dirty = sensors_dirty.copy()
outlier_idx = sensors_dirty.sample(n=30, random_state=5).index
sensors_dirty.loc[outlier_idx, "temperature"] = 999.0  # impossible sensor spike

print("\nRows with temperature == 999:", (sensors_dirty["temperature"] == 999.0).sum())

Total rows after duplication: 60050
Duplicate run_ids: 50

Unique shift_id values now: <StringArray>
['Morning', 'Evening', 'Night', 'Nite', 'EVENING', 'morning ']
Length: 6, dtype: str

Rows with temperature == 999: 30


In [7]:
sensors_dirty.to_csv("../data/raw/sensor_readings_dirty_demo.csv", index=False)
runs_dirty.to_csv("../data/raw/production_runs_dirty_demo.csv", index=False)
print("Dirty demo files saved.")

Dirty demo files saved.


In [8]:
print("Before cleaning:")
print("Missing temperature:", sensors_dirty["temperature"].isna().sum())
print("Missing vibration:", sensors_dirty["vibration"].isna().sum())

sensors_clean = sensors_dirty.copy()

# Strategy: fill missing sensor values with the median for that sensor
# Why median, not mean? Median is less affected by outliers (like our 999 spike)
temp_median = sensors_clean["temperature"].median()
vib_median = sensors_clean["vibration"].median()

sensors_clean["temperature"] = sensors_clean["temperature"].fillna(temp_median)
sensors_clean["vibration"] = sensors_clean["vibration"].fillna(vib_median)

print("\nAfter cleaning:")
print("Missing temperature:", sensors_clean["temperature"].isna().sum())
print("Missing vibration:", sensors_clean["vibration"].isna().sum())
print("\nTemperature median used:", temp_median)
print("Vibration median used:", vib_median)

Before cleaning:
Missing temperature: 1199
Missing vibration: 900

After cleaning:
Missing temperature: 0
Missing vibration: 0

Temperature median used: 74.45
Vibration median used: 4.2


In [9]:
print("Before cleaning:")
print("Total rows:", len(runs_dirty))
print("Duplicate run_ids:", runs_dirty["run_id"].duplicated().sum())

runs_clean = runs_dirty.copy()

# Strategy: drop duplicate run_ids, keeping the first occurrence
runs_clean = runs_clean.drop_duplicates(subset="run_id", keep="first")

print("\nAfter cleaning:")
print("Total rows:", len(runs_clean))
print("Duplicate run_ids:", runs_clean["run_id"].duplicated().sum())

Before cleaning:
Total rows: 60050
Duplicate run_ids: 50

After cleaning:
Total rows: 60000
Duplicate run_ids: 0


In [10]:
print("Before cleaning:")
print(runs_clean["shift_id"].unique())

# Strategy: standardize text (strip whitespace, fix casing), then map known variants
runs_clean["shift_id"] = runs_clean["shift_id"].str.strip().str.title()

# .str.title() turns "EVENING" -> "Evening", "morning" -> "Morning", but "Nite" stays "Nite"
# so we still need an explicit mapping for genuine abbreviations
shift_fix_map = {"Nite": "Night"}
runs_clean["shift_id"] = runs_clean["shift_id"].replace(shift_fix_map)

print("\nAfter cleaning:")
print(runs_clean["shift_id"].unique())
print(runs_clean["shift_id"].value_counts())

Before cleaning:
<StringArray>
['Morning', 'Evening', 'Night', 'Nite', 'EVENING', 'morning ']
Length: 6, dtype: str

After cleaning:
<StringArray>
['Morning', 'Evening', 'Night']
Length: 3, dtype: str
shift_id
Night      20124
Morning    19941
Evening    19935
Name: count, dtype: int64


In [11]:
print("Before cleaning:")
print("Rows with temperature == 999:", (sensors_clean["temperature"] == 999.0).sum())
print(sensors_clean["temperature"].describe())

# Strategy: treat 999 as a sensor error code, not a real reading -> replace with NaN, then fill with median
sensors_clean.loc[sensors_clean["temperature"] == 999.0, "temperature"] = np.nan
sensors_clean["temperature"] = sensors_clean["temperature"].fillna(sensors_clean["temperature"].median())

print("\nAfter cleaning:")
print("Rows with temperature == 999:", (sensors_clean["temperature"] == 999.0).sum())
print(sensors_clean["temperature"].describe())

Before cleaning:
Rows with temperature == 999: 30
count    60000.000000
mean        74.937219
std         21.362005
min         54.090000
25%         70.820000
50%         74.450000
75%         78.130000
max        999.000000
Name: temperature, dtype: float64

After cleaning:
Rows with temperature == 999: 0
count    60000.000000
mean        74.474944
std          5.400790
min         54.090000
25%         70.820000
50%         74.450000
75%         78.120000
max         97.840000
Name: temperature, dtype: float64


In [12]:
print("=== FINAL VALIDATION ===")
print("\nruns_clean:")
print("  Total rows:", len(runs_clean))
print("  Duplicate run_ids:", runs_clean["run_id"].duplicated().sum())
print("  Unique shifts:", runs_clean["shift_id"].unique())
print("  Missing values per column:\n", runs_clean.isna().sum()[runs_clean.isna().sum() > 0])

print("\nsensors_clean:")
print("  Total rows:", len(sensors_clean))
print("  Missing values per column:\n", sensors_clean.isna().sum()[sensors_clean.isna().sum() > 0])
print("  Temperature range:", sensors_clean["temperature"].min(), "-", sensors_clean["temperature"].max())
print("  Vibration range:", sensors_clean["vibration"].min(), "-", sensors_clean["vibration"].max())

# Save cleaned files — data/processed/, never overwriting data/raw/
runs_clean.to_csv("../data/processed/production_runs_clean.csv", index=False)
sensors_clean.to_csv("../data/processed/sensor_readings_clean.csv", index=False)
defects.to_csv("../data/processed/defect_events_clean.csv", index=False)   # unchanged, but copied for consistency
machines.to_csv("../data/processed/machines_clean.csv", index=False)

print("\nAll cleaned files saved to data/processed/")

=== FINAL VALIDATION ===

runs_clean:
  Total rows: 60000
  Duplicate run_ids: 0
  Unique shifts: <StringArray>
['Morning', 'Evening', 'Night']
Length: 3, dtype: str
  Missing values per column:
 Series([], dtype: int64)

sensors_clean:
  Total rows: 60000
  Missing values per column:
 Series([], dtype: int64)
  Temperature range: 54.09 - 97.84
  Vibration range: 0.0 - 9.37

All cleaned files saved to data/processed/
